<a href="https://colab.research.google.com/github/rifqiaditiya2006-cpu/Ant-Colony-Optimization-Using-Python/blob/main/Ant%20Colony%20Optimization%20Using%20Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Traveling Salesman Problem (TSP) — Ant Colony Optimization (ACO)
================================================================
Graf dari gambar: cari jalur terpendek H -> D, WAJIB melewati #

Simpul : #, A, B, C, D, E, F, G, H
Mulai  : H     Tujuan : D     Wajib  : #

Semua jalur valid H -> # -> D (brute force):
  Biaya 23: H -> G -> # -> C -> F -> E -> D  ← OPTIMAL
  Biaya 26: H -> E -> G -> # -> C -> B -> D
  Biaya 27: H -> G -> # -> C -> B -> D
  Biaya 30: H -> G -> # -> A -> C -> F -> E -> D
  Biaya 33: H -> E -> G -> # -> A -> C -> B -> D
  Biaya 34: H -> G -> # -> A -> C -> B -> D

Catatan ACO:
  Jalur optimal memerlukan semut memilih H->G (bobot=3) di langkah pertama,
  padahal H->E (bobot=1) jauh lebih menarik secara heuristik (η = 1/w).
  Ini adalah "greedy bias" inheren ACO pada graf dengan edge lokal sangat murah.
  Program menggunakan teknik Elite Ant + MMAS untuk memperbesar peluang optimal.
"""

import random
import math

In [ ]:
# DEFINISI GRAF

EDGES = [
    ('#', 'A', 3),
    ('#', 'C', 2),
    ('#', 'G', 5),
    ('A', 'C', 6),
    ('B', 'C', 9),
    ('B', 'D', 8),
    ('C', 'F', 4),
    ('D', 'E', 7),
    ('D', 'H', 9),
    ('E', 'F', 2),
    ('E', 'G', 1),
    ('E', 'H', 1),
    ('G', 'H', 3),
]

START   = 'H'
END     = 'D'
MUST    = '#'     # node wajib dilewati
OPTIMAL = 23      # jawaban optimal dari brute force

In [ ]:
# PARAMETER ACO  (dapat diubah)

N_ANTS      = 50      # jumlah semut per iterasi
N_ITER      = 300     # jumlah iterasi
ALPHA       = 2.0     # bobot feromon τ^α
BETA        = 2.0     # bobot heuristik (1/d)^β
RHO         = 0.2     # laju evaporasi feromon
Q           = 10.0    # konstanta deposit feromon
ELITE_COEF  = 5       # pengali deposit untuk jalur terbaik global (Elite Ant)
TAU_INIT    = 1.0     # feromon awal
TAU_MAX     = 5.0     # batas atas feromon (MMAS)
TAU_MIN     = 0.05    # batas bawah feromon (MMAS)
GREEDY_FRAC = 0.2     # fraksi semut yang berjalan greedy (diversifikasi)


In [ ]:
# BANGUN STRUKTUR DATA

def build_graph(edges):
    """Kembalikan adjacency dict: graph[u][v] = bobot."""
    graph = {}
    for a, b, w in edges:
        graph.setdefault(a, {})[b] = w
        graph.setdefault(b, {})[a] = w
    return graph


def build_pheromone(graph, tau_init):
    """Inisialisasi matriks feromon seragam."""
    return {u: {v: tau_init for v in graph[u]} for u in graph}

In [ ]:
# FUNGSI PEMBANTU

def path_cost(graph, path):
    """Total bobot jalur; kembalikan inf jika ada edge yang tidak ada."""
    total = 0
    for i in range(len(path) - 1):
        w = graph.get(path[i], {}).get(path[i + 1])
        if w is None:
            return math.inf
        total += w
    return total


def choose_next(current, candidates, graph, pheromone, alpha, beta):
    """
    Roulette-wheel selection.
    P(i→j) ∝ τ(i,j)^α × (1/w_ij)^β
    """
    scores = [
        (pheromone[current][n] ** alpha) * ((1.0 / graph[current][n]) ** beta)
        for n in candidates
    ]
    total = sum(scores)
    if total == 0:
        return random.choice(candidates)
    r = random.random()
    cumulative = 0.0
    for node, s in zip(candidates, scores):
        cumulative += s / total
        if r <= cumulative:
            return node
    return candidates[-1]


def choose_next_greedy(current, candidates, graph):
    """Pilih tetangga dengan bobot edge terkecil (greedy lokal)."""
    return min(candidates, key=lambda n: graph[current][n])


def ant_walk_segment(start, target, forbidden, graph, pheromone,
                     alpha, beta, greedy=False):
    """
    Semut berjalan dari `start` ke `target`.
    - forbidden : simpul yang tidak boleh dikunjungi ulang
    - greedy    : jika True, pakai choose_next_greedy (diversifikasi)
    Kembalikan jalur list atau None jika buntu.
    """
    visited = set(forbidden) | {start}
    path    = [start]
    current = start

    while current != target:
        candidates = [
            n for n in graph[current]
            if n not in visited or n == target
        ]
        if not candidates:
            return None

        if greedy:
            nxt = choose_next_greedy(current, candidates, graph)
        else:
            nxt = choose_next(current, candidates, graph, pheromone, alpha, beta)

        if nxt != target:
            visited.add(nxt)
        path.append(nxt)
        current = nxt

    return path


def ant_walk(start, end, must, graph, pheromone, alpha, beta, greedy=False):
    """
    Dua segmen:
      Segmen 1 : start → must     (bebas)
      Segmen 2 : must  → end      (forbidden = simpul seg1 kecuali must)

    Constraint ini memastikan:
      ✓ Jalur selalu melalui must
      ✓ Tidak ada simpul yang dikunjungi dua kali
    """
    seg1 = ant_walk_segment(
        start, must, set(),
        graph, pheromone, alpha, beta, greedy
    )
    if seg1 is None:
        return None

    # Forbidden untuk seg2 = semua yang sudah dikunjungi di seg1,
    # KECUALI must (ia adalah titik awal seg2, jadi tidak perlu diblokir)
    forbidden2 = set(seg1) - {must}
    seg2 = ant_walk_segment(
        must, end, forbidden2,
        graph, pheromone, alpha, beta, greedy
    )
    if seg2 is None:
        return None

    return seg1 + seg2[1:]    # must tidak muncul dua kali

In [ ]:
# UPDATE FEROMON  (MMAS style)

def evaporate(pheromone, rho, tau_min):
    """Evaporasi global: τ ← max(τ_min, τ × (1-ρ))."""
    for u in pheromone:
        for v in pheromone[u]:
            pheromone[u][v] = max(tau_min, pheromone[u][v] * (1 - rho))


def deposit_path(pheromone, path, delta, tau_max):
    """Deposit feromon sepanjang path sebesar delta, dibatasi tau_max."""
    for i in range(len(path) - 1):
        u, v = path[i], path[i + 1]
        pheromone[u][v] = min(tau_max, pheromone[u][v] + delta)
        pheromone[v][u] = min(tau_max, pheromone[v][u] + delta)

In [ ]:
# ALGORITMA ACO UTAMA

def aco_tsp(graph, start, end, must,
            n_ants=N_ANTS, n_iter=N_ITER,
            alpha=ALPHA, beta=BETA, rho=RHO, q=Q,
            elite_coef=ELITE_COEF,
            tau_init=TAU_INIT, tau_max=TAU_MAX, tau_min=TAU_MIN,
            greedy_frac=GREEDY_FRAC,
            verbose=True):
    """
    Jalankan ACO dengan teknik:
      • MMAS  : feromon dibatasi [tau_min, tau_max] agar eksplorasi tetap hidup
      • Elite Ant : jalur terbaik global mendapat deposit ekstra tiap iterasi
      • Greedy fraction : sebagian semut berjalan greedy untuk diversifikasi

    Kembalikan (best_path, best_cost, history).
    """
    pheromone        = build_pheromone(graph, tau_init)
    global_best_path = None
    global_best_cost = math.inf
    history          = []

    n_greedy = max(1, int(n_ants * greedy_frac))   # semut greedy per iterasi

    if verbose:
        print("=" * 66)
        print("  Ant Colony Optimization — TSP  (MMAS + Elite Ant)")
        print(f"  Jalur : {start} → {must} (wajib) → {end}")
        print(f"  Semut={n_ants}  Iterasi={n_iter}  α={alpha}  β={beta}  "
              f"ρ={rho}  Q={q}  Elite×{elite_coef}")
        print(f"  τ ∈ [{tau_min}, {tau_max}]  Semut greedy={n_greedy}/{n_ants}")
        print("=" * 66)

    for iteration in range(1, n_iter + 1):
        iter_results = []

        for ant_id in range(n_ants):
            # Sebagian kecil semut berjalan greedy untuk diversifikasi
            is_greedy = (ant_id < n_greedy)
            path = ant_walk(start, end, must, graph, pheromone,
                            alpha, beta, is_greedy)
            if path is None:
                continue
            cost = path_cost(graph, path)
            if not math.isinf(cost):
                iter_results.append((path, cost))
                if cost < global_best_cost:
                    global_best_cost = cost
                    global_best_path = path[:]

        # ── Update feromon ──
        evaporate(pheromone, rho, tau_min)

        # Deposit normal: semua semut yang berhasil
        for path, cost in iter_results:
            deposit_path(pheromone, path, q / cost, tau_max)

        # Elite deposit: jalur terbaik global mendapat bobot ekstra
        if global_best_path is not None:
            deposit_path(pheromone, global_best_path,
                         elite_coef * q / global_best_cost, tau_max)

        history.append((iteration, global_best_cost))

        # ── Progress log ──
        if verbose and (iteration % 30 == 0 or iteration == 1 or iteration == n_iter):
            ib    = min((c for _, c in iter_results), default=math.inf)
            ib_s  = str(ib) if not math.isinf(ib) else "—"
            flag  = "  ✓ OPTIMAL" if global_best_cost == OPTIMAL else ""
            print(f"  Iter {iteration:4d}/{n_iter} | "
                  f"Iterasi terbaik: {ib_s:>4} | "
                  f"Global terbaik: {global_best_cost}{flag}")

    return global_best_path, global_best_cost, history

In [ ]:
# CETAK HASIL

def print_result(graph, best_path, best_cost):
    print("\n" + "=" * 66)
    if best_path is None:
        print("  ❌  Tidak ada jalur valid yang ditemukan.")
        print("=" * 66)
        return

    label = "✅  OPTIMAL" if best_cost == OPTIMAL else "✅  HASIL TERBAIK"
    print(f"  {label}")
    print("=" * 66)

    detail = "  →  ".join(
        f"{best_path[i]}—({graph[best_path[i]][best_path[i+1]]})→{best_path[i+1]}"
        for i in range(len(best_path) - 1)
    )
    print(f"  Jalur   : {' → '.join(best_path)}")
    print(f"  Detail  : {detail}")
    print(f"  Panjang : {len(best_path) - 1} edge")
    print(f"  Total   : {best_cost}")
    print(f"\n  Verifikasi:")
    print(f"    Mulai dari {START}    : {'✓' if best_path[0]  == START else '✗'}")
    print(f"    Berakhir di {END}     : {'✓' if best_path[-1] == END   else '✗'}")
    print(f"    Melewati {MUST}       : {'✓' if MUST in best_path       else '✗'}")
    gap = best_cost - OPTIMAL
    gp  = f"+{gap}" if gap > 0 else "0"
    print(f"\n  Optimal (brute force) : {OPTIMAL}")
    print(f"  Gap ACO               : {gp}  ({gap/OPTIMAL*100:.1f}%)")
    print("=" * 66)

In [ ]:
# MULTI-RUN — statistik stabilitas

def multi_run(graph, n_runs=10):
    print("\n" + "=" * 66)
    print(f"  MULTI-RUN  ({n_runs} percobaan, seed acak)")
    print("=" * 66)
    results = []
    for run in range(1, n_runs + 1):
        random.seed()   # seed acak tiap run
        path, cost, _ = aco_tsp(
            graph, START, END, MUST, verbose=False,
            n_ants=N_ANTS, n_iter=N_ITER,
            alpha=ALPHA, beta=BETA, rho=RHO, q=Q,
            elite_coef=ELITE_COEF,
        )
        results.append((cost, path))
        label    = " → ".join(path) if path else "—"
        opt_flag = "  ← OPTIMAL ✓" if cost == OPTIMAL else ""
        print(f"  Run {run:2d}: biaya={cost}   [{label}]{opt_flag}")

    valid = [c for c, _ in results if not math.isinf(c)]
    if valid:
        n_opt = sum(1 for c in valid if c == OPTIMAL)
        print(f"\n  Rata-rata      : {sum(valid)/len(valid):.2f}")
        print(f"  Minimum        : {min(valid)}")
        print(f"  Maksimum       : {max(valid)}")
        print(f"  Temukan optimal: {n_opt}/{n_runs} run ({n_opt/n_runs*100:.0f}%)")
    print("=" * 66)

In [ ]:
# MAIN

if __name__ == "__main__":
    random.seed(7)    # ganti atau hapus untuk hasil acak berbeda

    graph = build_graph(EDGES)

    best_path, best_cost, history = aco_tsp(
        graph, START, END, MUST,
        n_ants      = N_ANTS,
        n_iter      = N_ITER,
        alpha       = ALPHA,
        beta        = BETA,
        rho         = RHO,
        q           = Q,
        elite_coef  = ELITE_COEF,
        tau_init    = TAU_INIT,
        tau_max     = TAU_MAX,
        tau_min     = TAU_MIN,
        greedy_frac = GREEDY_FRAC,
        verbose     = True,
    )

    print_result(graph, best_path, best_cost)

    print()
    ans = input("Jalankan multi-run (10x) untuk statistik? [y/N]: ").strip().lower()
    if ans == 'y':
        multi_run(graph, n_runs=10)

  Ant Colony Optimization — TSP  (MMAS + Elite Ant)
  Jalur : H → # (wajib) → D
  Semut=50  Iterasi=300  α=2.0  β=2.0  ρ=0.2  Q=10.0  Elite×5
  τ ∈ [0.05, 5.0]  Semut greedy=10/50
  Iter    1/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter   30/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter   60/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter   90/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  120/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  150/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  180/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  210/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  240/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  270/300 | Iterasi terbaik:   26 | Global terbaik: 26
  Iter  300/300 | Iterasi terbaik:   26 | Global terbaik: 26

  ✅  HASIL TERBAIK
  Jalur   : H → E → G → # → C → B → D
  Detail  : H—(1)→E  →  E—(1)→G  →  G—(5)→#  →  #—(2)→C  →  C—(9)→B  →  B—(8)→D
  Panjang :